# Biological Validation (lightweight single-cell RNA guidance)

This notebook provides a beginner-friendly, lightweight workflow to explore single-cell signals relevant to PCOS phenotypes. It is *not* a full bioinformatics pipeline — the goal is marker genes, pathway summaries, UMAP visuals, and clinician-friendly summaries.

In [ ]:
# Install dependencies in Colab if needed (uncomment in Colab)
# !pip install scanpy anndata scikit-learn matplotlib seaborn gseapy pandas

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

sc.settings.verbosity = 2
np.random.seed(42)

In [ ]:
# Example: load a small public scRNA-seq AnnData (or provide your own)
# Replace the following with a real file path or a public dataset in Colab
# adata = sc.read_h5ad('/path/to/small_sample.h5ad')

# For demonstration we'll create a tiny synthetic AnnData with marker-like patterns
n_cells = 500
genes = ['IL6','TNF','CXCL8','INSR','AKT1','AKT2','AR','SRD5A1','AMH','CYP19A1','FOXL2','FSHR']
X = np.random.poisson(1.0, size=(n_cells, len(genes))).astype(float)
# inject signal for a subset of cells to mimic inflammatory/insulin/androgen signals
X[:120, 0:3] += np.random.poisson(5, size=(120,3))   # inflammatory genes up in group A
X[120:260, 3:6] += np.random.poisson(6, size=(140,3)) # insulin pathway up in group B
X[260:380, 6:9] += np.random.poisson(4, size=(120,3)) # androgen signals up in group C

import anndata as ad
adata = ad.AnnData(X)
adata.var['gene_symbols'] = genes
adata.var_names = genes

# Basic preprocessing and UMAP
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=10)
sc.pp.scale(adata)
sc.tl.pca(adata, n_comps=20)
sc.pp.neighbors(adata)
sc.tl.umap(adata)

# plot UMAP colored by synthetic signal groups
adata.obs['group'] = ['inflam']*120 + ['insulin']*140 + ['androgen']*120 + ['control']*(n_cells-380)
sc.pl.umap(adata, color='group', title='Synthetic UMAP: inflammatory / insulin / androgen / control', show=False)
plt.show()

In [ ]:
# Identify simple marker genes per group using t-test (demo)
sc.tl.rank_genes_groups(adata, 'group', method='t-test')
markers = {}
for grp in adata.obs['group'].unique():
    df = sc.get.rank_genes_groups_df(adata, group=grp)
    markers[grp] = df[['names','logfoldchanges','pvals_adj']].head(6)
    print('Top markers for', grp)
    display(markers[grp])

## Clinician-friendly pathway summaries
The notebook maps marker genes to pathways of interest (inflammatory, insulin signaling, androgen signaling, ovarian function). Below is an example summary format to include in reports:

- Observed inflammatory pathway activity (IL6, TNF, CXCL8) elevated in a cell subset — suggests immune/inflammatory contribution.
- Insulin signaling components (INSR, AKT1/2) show higher expression in a separate subset — aligns with insulin-resistance phenotype.
- Androgen-related genes (AR, SRD5A1) show localized enrichment — supports hyperandrogenism signal.
- Ovarian markers (AMH, FSHR, FOXL2) elevated in follicular-like cells — consistent with ovarian dysregulation.

Clinician summary (required phrasing):

Observed inflammatory and insulin-signaling pathway dysregulation aligns with the patient's predicted phenotype.